# DSA5208 · Cassandra 会话一致性实验

以本目录的代码为准：三个 Cassandra 节点，RF=3，比较 RYW、MR、MW、WFR。

**展示路线**：环境与架构 → 参数与预测 → 正常场景 → 节点故障 → 网络分区 → 历史结果 → 原始证据与局限。

使用 **Shift + Enter** 运行当前单元格。默认“Run All”只加载文件、显示按钮和历史结果；集群操作需要点击相应按钮。先用每组 5 次迭代演示，再按需增加。

In [ ]:
from pathlib import Path
import sys, os, subprocess, time, json, csv, html
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, Code, clear_output

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "experiments/common.py").exists() and (p / "docker-compose.yml").exists()), None)
if ROOT is None:
    raise RuntimeError("请从 Cassandra-Lab-main 项目中启动 Notebook。")
docker_bin = Path("/Applications/Docker.app/Contents/Resources/bin")
if docker_bin.exists():
    os.environ["PATH"] = str(docker_bin) + os.pathsep + os.environ.get("PATH", "")
print("项目:", ROOT)
print("Python:", sys.executable)
print("Notebook 复用 experiments/ 和 scripts/ 中的实现。")


## 1. 项目要求与架构

| 内容 | 本项目实现 |
|---|---|
| 复制数据库 | cass1、cass2、cass3；每个容器一个逻辑节点 |
| 副本放置 | SimpleStrategy，RF=3；三节点均保存每个 partition 的副本 |
| 客户端入口 | Mac 的 9042 / 9043 / 9044 → 各容器 9042 |
| 节点间通信 | 7000；延迟和分区作用于此流量 |
| 自变量 | 写入 CL、读取 CL、故障场景、网络延迟 |
| 指标 | 有效样本、违例数、失败迭代数及各自比率 |

Coordinator 是每次请求的协调节点，不是永久主节点。客户端连接某节点不代表它只访问该节点上的副本。

三个容器共享同一台 Mac，不能据此证明物理机器级容错。

In [ ]:
display(Code((ROOT / "docker-compose.yml").read_text(), language="yaml"))
display(Code((ROOT / "scripts/setup_keyspace.cql").read_text(), language="sql"))
env_file = ROOT / "results/environment.txt"
if env_file.exists():
    display(Markdown("### 历史环境快照（不代表当前运行环境）"))
    print(env_file.read_text())


## 2. 一致性配置与预测

N=3；ONE 需要 1 个副本响应，QUORUM 需要 2 个，ALL 需要 3 个。

| W / R | W+R>N | 在无并发、时间戳有序、写入成功的受控历史下 |
|---|---|---|
| ONE / ONE | 否 | 可能出现旧读；不保证每次出现 |
| ONE / QUORUM | 否 | 读写集合可能不相交 |
| QUORUM / QUORUM | 是 | 有读写交集 |
| ALL / ONE | 是 | 有读写交集，但写入需要全部副本可达 |

RYW：写后能读到自己的写或更新版本。MR：连续读取不倒退。
MW：同一会话写入顺序。WFR：读后写应保持对已读写入的依赖。

**解释边界**：本项目 MW/WFR 主要检查同一 key 的最终覆盖值，以及人为时间戳倒序；零违例不能证明全部副本的完整因果应用顺序。read_repair=NONE 是实验配置，结论需注明该前提。

## 3. 现场演示工具

下面的按钮会真实操作 Docker 或运行原脚本，并在单元格下方实时显示输出。运行实验前先启动 Docker Desktop。

长任务可通过 Notebook 的“Interrupt Kernel”中断。故障操作后使用后面的恢复按钮。历史结果展示不依赖 Docker。

In [ ]:
def run_cmd(argv, timeout=None):
    print("$", " ".join(map(str, argv)), flush=True)
    proc = subprocess.Popen(list(map(str, argv)), cwd=ROOT,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, start_new_session=True)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        rc = proc.wait(timeout=timeout)
        if rc:
            raise RuntimeError(f"命令失败，退出码 {rc}；请查看上方输出。")
    except BaseException:
        if proc.poll() is None:
            import signal
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait()
        raise

def action(label, callback):
    button = widgets.Button(description=label, layout=widgets.Layout(width="auto"))
    output = widgets.Output()
    def invoke(_):
        button.disabled = True
        with output:
            clear_output(wait=True)
            try:
                callback()
            except Exception as e:
                print(f"未完成：{type(e).__name__}: {e}")
        button.disabled = False
    button.on_click(invoke)
    display(widgets.VBox([button, output]))

def script(name):
    run_cmd(["bash", ROOT / "scripts" / name])

action("检查 Docker 与容器状态", lambda: run_cmd(["docker", "compose", "ps"]))
action("启动三节点并初始化（约数分钟）", lambda: script("bootstrap.sh"))
action("查看三节点状态", lambda: run_cmd(["docker", "exec", "cass1", "nodetool", "status"]))


In [ ]:
run_cmd([
    sys.executable, "-c",
    "import sys; sys.path.insert(0, 'experiments'); "
    "from common import connect_nodes; "
    "nodes = connect_nodes(); "
    "[node.shutdown() for node in nodes.values()]; "
    "print('三个节点连接成功')"
])

## 4. 正常运行与单组实验

可先清除延迟做基线，再注入默认 200ms ± 50ms 延迟重复比较。延迟参数以脚本和结果中的实际记录为准。

下方直接运行现有实验程序。节点分配与 run_matrix.sh 一致。每次执行使用独立结果目录，避免覆盖历史文件，并对本次运行使用唯一标识；程序原始判定逻辑保持不变。

In [ ]:
action("清除网络规则：无延迟基线", lambda: script("clear_latency.sh"))
action("注入默认复制延迟", lambda: script("inject_latency.sh"))

scenario = widgets.Dropdown(options=["normal", "node_failure", "partition_majority", "partition_minority"], description="场景")
model = widgets.Dropdown(options=["ryw", "mr", "mw", "wfr"], description="模型")
wcl = widgets.Dropdown(options=["ONE", "QUORUM", "ALL"], description="写 CL")
rcl = widgets.Dropdown(options=["ONE", "QUORUM", "ALL"], description="读 CL")
mode = widgets.Dropdown(options=["natural", "skew", "both"], description="MW/WFR")
iterations = widgets.BoundedIntText(value=5, min=1, max=10000, description="迭代次数")
display(widgets.VBox([scenario, model, wcl, rcl, mode, iterations]))

def experiment_args():
    sc, m = scenario.value, model.value
    if sc == "normal":
        nodes = "1,2,3"
        roles = {"ryw": [], "mr": [], "mw": [], "wfr": []}
    elif sc in ("node_failure", "partition_majority"):
        nodes = "1,2"
        roles = {
            "ryw": ["--write-node","1","--read-node","2"],
            "mr": ["--seed-cl","QUORUM"],
            "mw": ["--write1-node","1","--write2-node","2","--verify-cl","QUORUM","--settle-ms","1000"],
            "wfr": ["--client-node","1","--setup-cl","QUORUM","--verify-cl","QUORUM","--settle-ms","1000"]}
    else:
        nodes = "3"
        roles = {
            "ryw": ["--write-node","3","--read-node","3"],
            "mr": ["--write-node","3","--read1-node","3","--read2-node","3","--seed-cl","ONE"],
            "mw": ["--write1-node","3","--write2-node","3","--verify-node","3","--verify-cl","ONE","--settle-ms","1000"],
            "wfr": ["--write-node","3","--read-node","3","--client-node","3","--verify-node","3","--setup-cl","ONE","--verify-cl","ONE","--settle-ms","1000"]}
    args = ["--scenario", "partition" if sc.startswith("partition") else sc,
            "--nodes", nodes, "--write-cl", wcl.value, "--read-cl", rcl.value,
            "-n", str(iterations.value)] + roles[m]
    if m in ("mw", "wfr"):
        args += ["--mode", mode.value]
    return args

def run_selected():
    run_cmd([sys.executable, ROOT / "notebooks/run_demo.py",
             model.value, *experiment_args()])

action("运行所选单组实验", run_selected)


## 5. 节点故障

顺序：正常基线 → 停止 cass3 → 上面选择 node_failure → 运行各 CL → 恢复 cass3。

故障稳定后 ONE/QUORUM 可能成功，ALL 无法满足。失败可能表现为 Unavailable 或 Timeout。失败迭代不能当作一致性违例或通过。

In [ ]:
action("停止 cass3", lambda: run_cmd(["docker", "stop", "cass3"]))
action("启动 cass3 并等待健康", lambda: run_cmd(["docker", "compose", "up", "-d", "--wait", "--wait-timeout", "600", "cass3"]))


## 6. 网络分区

先保证三个节点健康，再注入延迟规则，最后隔离 cass3。选择 partition_majority 和 partition_minority 分别运行。

少数派矩阵的读写都落在 cass3，主要展示可用性；它没有测试同一客户端跨分区迁移的全部会话历史。

In [ ]:
action("隔离 cass3（先运行延迟注入）", lambda: run_cmd(["bash", ROOT / "scripts/partition.sh", "cass3"]))
action("恢复分区通信（保留延迟）", lambda: script("heal_partition.sh"))
action("恢复后检查集群", lambda: run_cmd(["docker", "exec", "cass1", "nodetool", "status"]))


## 7. 汇总结果与图表

同时读取历史 results/all_results.csv 和 Notebook 独立运行目录。每行保留来源，不把不同延迟、场景或配置混成一个平均值。

违例率 = violations / triggered；失败迭代率 = errors / iterations。triggered=0 时违例率为 N/A。

点击“刷新结果”，新实验完成后再次点击即可看到新增记录。

In [ ]:
def load_results():
    paths = [ROOT / "results/all_results.csv", *sorted((ROOT / "results/notebook_runs").glob("*/all_results.csv"))]
    frames = []
    for path in paths:
        if path.exists() and path.stat().st_size:
            frame = pd.read_csv(path)
            frame["source"] = str(path.relative_to(ROOT))
            frame["source_row"] = range(2, len(frame) + 2)
            frames.append(frame)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    for col in ["iterations", "triggered", "violations", "errors"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["observed_violation_rate"] = df["violations"].div(df["triggered"].where(df["triggered"] > 0))
    df["failed_iteration_rate"] = df["errors"].div(df["iterations"].where(df["iterations"] > 0))
    df["assessment"] = "未观察到违反"
    df.loc[df["violations"] > 0, "assessment"] = "观察到违反"
    df.loc[df["triggered"].fillna(0) == 0, "assessment"] = "N/A：无有效样本"
    return df

def show_results():
    global results_df
    results_df = load_results()
    if results_df.empty:
        print("当前没有可读取的汇总数据。")
        return
    cols = ["run_ts","model","scenario","mode","write_cl","read_cl","verify_cl",
            "delay_ms","jitter_ms","iterations","triggered","violations","errors",
            "observed_violation_rate","failed_iteration_rate","assessment","topology","source","source_row"]
    display(results_df[[c for c in cols if c in results_df]].tail(30))
    # 展示最近 12 次运行，逐行绘制，不跨配置聚合。
    recent = results_df.tail(12).copy()
    recent["label"] = [f"{i+1}: {r.model}/{r.scenario}/{r.write_cl}-{r.read_cl}/{r['mode']}"
                       for i, (_,r) in enumerate(recent.iterrows())]
    fig, axes = plt.subplots(1, 2, figsize=(13, max(4, len(recent)*0.4)), sharey=True)
    for ax, col, title in zip(axes,
            ["observed_violation_rate","failed_iteration_rate"],
            ["Observed violations / eligible iterations", "Failed iterations / attempted iterations"]):
        ax.barh(recent["label"], recent[col].fillna(0))
        ax.set_xlim(0, 1.08)
        ax.set_title(title)
        for y, value in enumerate(recent[col]):
            ax.text(0.02 if pd.isna(value) else value+0.01, y,
                    "N/A" if pd.isna(value) else f"{value:.1%}", va="center")
    axes[0].invert_yaxis()
    plt.tight_layout()
    plt.show()
    print("完整参数、延迟和来源见上表；图中每个条目对应独立运行。")

action("刷新结果", show_results)
show_results()


## 8. 检查一条原始证据

选择 JSONL 文件查看前五条。重点解释写入/读取值、coordinator、时间戳、triggered、violation 和 error_type。

历史 run_id 仅有时分秒，不能保证跨日期唯一；本 Notebook 新运行采用 UUID，日志隔离保存。

In [ ]:
trace_paths = sorted((ROOT / "results").rglob("*.jsonl"))
trace_select = widgets.Dropdown(options=[(str(p.relative_to(ROOT)), str(p)) for p in trace_paths],
                               description="原始日志", layout=widgets.Layout(width="95%"))
display(trace_select)
def show_trace():
    if not trace_select.value:
        print("暂无 JSONL 文件；完成实验后重新运行此单元格。")
        return
    with open(trace_select.value) as f:
        for _, line in zip(range(5), f):
            print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))
action("查看前五条记录", show_trace)


## 9. 结论如何陈述

1. 根据预期与观测解释各配置的读写交集、旧读和失败情况。
2. 0 次违反只说明有限运行未观察到反例；N/A 表示没有有效样本。
3. MW/WFR 的最终覆盖测试不能替代完整的副本操作顺序验证。
4. ALL 读会协调副本返回值，但 read_repair=NONE 时不等于把所有副本修复一致。
5. 一秒等待不是收敛证明；拓扑检查存在边界，展示时核对各节点视图。
6. read_repair=NONE 和 speculative_retry=NONE 的结论需限定配置。可增加默认配置对照。

参考：[作业代码 README](../README.md)、[Cassandra 4.1 架构](https://cassandra.apache.org/doc/4.1/cassandra/architecture/dynamo.html)、[Read repair](https://cassandra.apache.org/doc/4.1/cassandra/operating/read_repair.html)。

## 10. 演示结束

先恢复 cass3，再恢复分区与清除延迟，确认节点状态。停止容器会保留容器数据。

In [ ]:
action("停止全部项目容器（保留数据）", lambda: run_cmd(["docker", "compose", "stop"]))
